# 🗺️ Otimização de Rotas Comerciais — Do Caos à Estratégia
**Distribuidora Farmacêutica · Grande São Paulo · Python + Algoritmos de Otimização**

---

## 🎯 Contexto de Negócio

Uma distribuidora farmacêutica com atuação na Grande São Paulo possui **10 representantes comerciais** responsáveis por visitar **50 farmácias** semanalmente.

O problema: rotas planejadas no feeling, sem critério geográfico — gerando deslocamentos desnecessários, cobertura irregular e alto custo operacional.

**Este projeto implementa e compara múltiplos algoritmos de otimização**, desde heurísticas clássicas até melhorias iterativas, com análise completa de saving, ROI e planejamento semanal.

| | |
|--|--|
| **Farmácias** | 50 pontos de venda na Grande SP |
| **Representantes** | 10 profissionais de campo |
| **Algoritmos comparados** | Aleatório vs Nearest Neighbor vs 2-Opt vs Prioridade |
| **Entregável** | Planejamento semanal completo com ROI projetado |

---

## 0. Imports e Configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import folium
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
import os
import random
import itertools
import time

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

os.makedirs('images', exist_ok=True)
os.makedirs('mapas', exist_ok=True)

# Paleta do projeto
CORES = {
    'antes':    '#d62728',
    'nn':       '#1a6b3c',
    'opt2':     '#f5c518',
    'prior':    '#2196F3',
    'cluster':  '#9467bd',
    'destaque': '#ff7f0e',
}

CORES_REPS = [
    '#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
    '#a65628','#f781bf','#555555','#66c2a5','#fc8d62'
]

print('✅ Ambiente configurado!')
print(f'   NumPy {np.__version__} | Pandas {pd.__version__} | Sklearn OK | Folium OK')

## 1. Geração dos Dados Sintéticos

In [ ]:
regioes_sp = {
    'Centro':       (-23.5505, -46.6333),
    'Paulista':     (-23.5613, -46.6560),
    'Faria Lima':   (-23.5763, -46.6872),
    'Berrini':      (-23.5960, -46.6875),
    'Tatuapé':      (-23.5402, -46.5766),
    'Santo André':  (-23.6629, -46.5280),
    'Guarulhos':    (-23.4543, -46.5333),
    'Osasco':       (-23.5325, -46.7919),
    'Santana':      (-23.4975, -46.6261),
    'Mooca':        (-23.5530, -46.6019),
}

N_FARMACIAS      = 50
N_REPRESENTANTES = 10
CUSTO_KM         = 1.50  # R$/km
MAX_VISITAS_DIA  = 1     # capacidade diária por representante (1 farmácia/dia → distribuição uniforme na semana)

# Gera farmácias
farmacias = []
for i in range(N_FARMACIAS):
    regiao = random.choice(list(regioes_sp.keys()))
    lat_b, lon_b = regioes_sp[regiao]
    prioridade = np.random.choice(['Alta', 'Media', 'Baixa'], p=[0.25, 0.45, 0.30])
    farmacias.append({
        'id': f'FM{i+1:03d}',
        'nome': f'Farmácia {regiao} {i+1}',
        'regiao': regiao,
        'lat': lat_b + np.random.uniform(-0.05, 0.05),
        'lon': lon_b + np.random.uniform(-0.05, 0.05),
        'prioridade': prioridade,
        'peso_prior': {'Alta': 3, 'Media': 2, 'Baixa': 1}[prioridade],
        'pedidos_mes': np.random.randint(20, 200),
        'ticket_medio': np.random.randint(500, 5000),
    })

df_farm = pd.DataFrame(farmacias)
df_farm['faturamento'] = df_farm['pedidos_mes'] * df_farm['ticket_medio']

# Gera representantes
representantes = []
for i in range(N_REPRESENTANTES):
    regiao = random.choice(list(regioes_sp.keys()))
    lat_b, lon_b = regioes_sp[regiao]
    representantes.append({
        'id': f'REP{i+1:02d}',
        'nome': f'Representante {i+1}',
        'lat_base': lat_b + np.random.uniform(-0.02, 0.02),
        'lon_base': lon_b + np.random.uniform(-0.02, 0.02),
        'regiao_base': regiao
    })

df_reps = pd.DataFrame(representantes)

print(f'✅ {N_FARMACIAS} farmácias geradas | {N_REPRESENTANTES} representantes')
print(f'\n📊 Farmácias por prioridade:')
print(df_farm['prioridade'].value_counts().to_string())
print(f'\n💊 Faturamento mensal total: R$ {df_farm["faturamento"].sum():,.0f}')
print(f'   Alta prioridade: R$ {df_farm[df_farm["prioridade"]=="Alta"]["faturamento"].sum():,.0f}')
df_farm.head()

## 2. Funções Base de Cálculo
Antes de implementar os algoritmos, definimos as funções utilitárias compartilhadas.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Distância em km entre dois pontos geográficos (Haversine)."""
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def dist_rota(pontos):
    """Distância total de uma rota (lista de tuplas lat/lon)."""
    return sum(
        haversine(pontos[i][0], pontos[i][1], pontos[i+1][0], pontos[i+1][1])
        for i in range(len(pontos) - 1)
    )

def matriz_distancias(pontos):
    """Matriz NxN de distâncias entre todos os pontos."""
    n = len(pontos)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = haversine(pontos[i][0], pontos[i][1], pontos[j][0], pontos[j][1])
            M[i][j] = M[j][i] = d
    return M

# Distribui farmácias entre representantes (fixo para todos os algoritmos)
farm_idx_por_rep = np.array_split(df_farm.sample(frac=1, random_state=42).index, N_REPRESENTANTES)

print('✅ Funções de cálculo definidas')
print(f'   Farmácias por representante: {[len(x) for x in farm_idx_por_rep]}')

## 3. Benchmark — Rota Aleatória (Situação Atual)
Simula o planejamento manual sem critério geográfico.

In [ ]:
def rota_aleatoria(base, pontos):
    """Ordem aleatória de visitas — simula o planejamento manual."""
    rota = [base] + random.sample(pontos, len(pontos)) + [base]
    return rota

rotas_aleatorio = {}
dist_aleatorio  = {}

for i, rep in df_reps.iterrows():
    base   = (rep['lat_base'], rep['lon_base'])
    pontos = [(df_farm.loc[j,'lat'], df_farm.loc[j,'lon']) for j in farm_idx_por_rep[i]]
    rota   = rota_aleatoria(base, pontos)
    rotas_aleatorio[rep['id']] = rota
    dist_aleatorio[rep['id']]  = dist_rota(rota)

total_aleatorio = sum(dist_aleatorio.values())
print(f'🎲 Rota ALEATÓRIA (situação atual):')
print(f'   Total: {total_aleatorio:.1f} km | Média: {total_aleatorio/N_REPRESENTANTES:.1f} km/rep')
print(f'   Custo semanal: R$ {total_aleatorio * CUSTO_KM:,.2f}')

## 4. Algoritmo 1 — Nearest Neighbor (NN)
A cada passo visita o ponto não visitado mais próximo. Heurística clássica de TSP.

In [ ]:
def nearest_neighbor(base, pontos):
    """Nearest Neighbor — sempre vai para o ponto mais próximo não visitado."""
    rota = [base]
    nao_vis = list(range(len(pontos)))
    atual = base
    while nao_vis:
        dists = [haversine(atual[0], atual[1], pontos[j][0], pontos[j][1]) for j in nao_vis]
        prox  = nao_vis[np.argmin(dists)]
        atual = pontos[prox]
        rota.append(atual)
        nao_vis.remove(prox)
    return rota + [base]

rotas_nn = {}
dist_nn  = {}

for i, rep in df_reps.iterrows():
    base   = (rep['lat_base'], rep['lon_base'])
    pontos = [(df_farm.loc[j,'lat'], df_farm.loc[j,'lon']) for j in farm_idx_por_rep[i]]
    rota   = nearest_neighbor(base, pontos)
    rotas_nn[rep['id']] = rota
    dist_nn[rep['id']]  = dist_rota(rota)

total_nn = sum(dist_nn.values())
saving_nn = total_aleatorio - total_nn
print(f'🟢 Nearest Neighbor:')
print(f'   Total: {total_nn:.1f} km | Média: {total_nn/N_REPRESENTANTES:.1f} km/rep')
print(f'   Saving vs aleatório: {saving_nn:.1f} km ({saving_nn/total_aleatorio*100:.1f}%)')
print(f'   Economia semanal: R$ {saving_nn * CUSTO_KM:,.2f}')

## 5. Algoritmo 2 — 2-Opt (Melhoria Iterativa)
Parte da rota do NN e melhora trocando pares de arestas até não haver mais melhoria. Geralmente encontra soluções 5-15% melhores que o NN.

In [ ]:
def two_opt(rota):
    """
    2-Opt: testa todas as trocas possíveis de segmentos.
    Se inverter um segmento melhora a distância total, aceita a melhoria.
    Repete até não haver mais melhorias possíveis.
    """
    melhor = rota[:]
    melhorou = True
    while melhorou:
        melhorou = False
        for i in range(1, len(melhor) - 2):
            for j in range(i + 1, len(melhor) - 1):
                nova = melhor[:i] + melhor[i:j+1][::-1] + melhor[j+1:]
                if dist_rota(nova) < dist_rota(melhor):
                    melhor = nova
                    melhorou = True
    return melhor

rotas_2opt = {}
dist_2opt  = {}

print('⏳ Executando 2-Opt (pode levar alguns segundos)...')
t0 = time.time()

for i, rep in df_reps.iterrows():
    rota_inicial = rotas_nn[rep['id']]
    rota_otim    = two_opt(rota_inicial)
    rotas_2opt[rep['id']] = rota_otim
    dist_2opt[rep['id']]  = dist_rota(rota_otim)

total_2opt  = sum(dist_2opt.values())
saving_2opt = total_aleatorio - total_2opt
melhora_vs_nn = total_nn - total_2opt

print(f'\n🟡 2-Opt (tempo: {time.time()-t0:.1f}s):')
print(f'   Total: {total_2opt:.1f} km | Média: {total_2opt/N_REPRESENTANTES:.1f} km/rep')
print(f'   Saving vs aleatório: {saving_2opt:.1f} km ({saving_2opt/total_aleatorio*100:.1f}%)')
print(f'   Melhora vs NN: {melhora_vs_nn:.1f} km ({melhora_vs_nn/total_nn*100:.1f}%)')
print(f'   Economia semanal: R$ {saving_2opt * CUSTO_KM:,.2f}')

## 6. Algoritmo 3 — Rota com Prioridade de Visita
Garante que farmácias de alta prioridade sejam visitadas primeiro, independente da distância.

In [ ]:
def nearest_neighbor_prioridade(base, pontos_info):
    """
    NN com prioridade: visita todos os pontos de alta prioridade primeiro
    (também por NN dentro do grupo), depois média, depois baixa.
    pontos_info: lista de dicts com 'coords' e 'prioridade'
    """
    def nn_grupo(inicio, grupo):
        if not grupo:
            return [], inicio
        rota_g = []
        nao_vis = list(range(len(grupo)))
        atual = inicio
        while nao_vis:
            dists = [haversine(atual[0], atual[1], grupo[j][0], grupo[j][1]) for j in nao_vis]
            prox  = nao_vis[np.argmin(dists)]
            atual = grupo[prox]
            rota_g.append(atual)
            nao_vis.remove(prox)
        return rota_g, atual

    alta  = [p['coords'] for p in pontos_info if p['prioridade'] == 'Alta']
    media = [p['coords'] for p in pontos_info if p['prioridade'] == 'Media']
    baixa = [p['coords'] for p in pontos_info if p['prioridade'] == 'Baixa']

    rota = [base]
    atual = base
    for grupo in [alta, media, baixa]:
        trecho, atual = nn_grupo(atual, grupo)
        rota.extend(trecho)
    return rota + [base]

rotas_prior = {}
dist_prior  = {}

for i, rep in df_reps.iterrows():
    base = (rep['lat_base'], rep['lon_base'])
    pontos_info = [
        {'coords': (df_farm.loc[j,'lat'], df_farm.loc[j,'lon']),
         'prioridade': df_farm.loc[j,'prioridade']}
        for j in farm_idx_por_rep[i]
    ]
    rota = nearest_neighbor_prioridade(base, pontos_info)
    rotas_prior[rep['id']] = rota
    dist_prior[rep['id']]  = dist_rota(rota)

total_prior  = sum(dist_prior.values())
saving_prior = total_aleatorio - total_prior

print(f'🔵 Rota com Prioridade:')
print(f'   Total: {total_prior:.1f} km | Média: {total_prior/N_REPRESENTANTES:.1f} km/rep')
print(f'   Saving vs aleatório: {saving_prior:.1f} km ({saving_prior/total_aleatorio*100:.1f}%)')
print(f'   Economia semanal: R$ {saving_prior * CUSTO_KM:,.2f}')
print(f'\n   ✅ Garante: Alta prioridade sempre visitada primeiro')

## 7. Clusterização por Região (K-Means)
Agrupa farmácias geograficamente antes de atribuir representantes, garantindo territórios coesos sem cruzamento de rotas.

In [ ]:
# Aplica K-Means para criar territórios geográficos coesos
coords = df_farm[['lat', 'lon']].values
kmeans = KMeans(n_clusters=N_REPRESENTANTES, random_state=42, n_init=10)
df_farm['cluster'] = kmeans.fit_predict(coords)

# Reatribui farmácias por cluster
farm_idx_cluster = [df_farm[df_farm['cluster'] == c].index.tolist() for c in range(N_REPRESENTANTES)]

rotas_cluster = {}
dist_cluster  = {}

for i, rep in df_reps.iterrows():
    base   = (rep['lat_base'], rep['lon_base'])
    pontos = [(df_farm.loc[j,'lat'], df_farm.loc[j,'lon']) for j in farm_idx_cluster[i]]
    if pontos:
        rota = nearest_neighbor(base, pontos)
        rota = two_opt(rota)
    else:
        rota = [base, base]
    rotas_cluster[rep['id']] = rota
    dist_cluster[rep['id']]  = dist_rota(rota)

total_cluster  = sum(dist_cluster.values())
saving_cluster = total_aleatorio - total_cluster

# Visualização dos clusters
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Sem cluster
cores_pri = {'Alta': '#d62728', 'Media': '#ff7f0e', 'Baixa': '#2196F3'}
for _, f in df_farm.iterrows():
    axes[0].scatter(f['lon'], f['lat'], c=cores_pri[f['prioridade']], s=60, alpha=0.8, zorder=3)
axes[0].set_title('Distribuição Original\n(sem territórios definidos)', fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
patches = [mpatches.Patch(color=c, label=p) for p, c in cores_pri.items()]
axes[0].legend(handles=patches, title='Prioridade')

# Com cluster
for c in range(N_REPRESENTANTES):
    subset = df_farm[df_farm['cluster'] == c]
    axes[1].scatter(subset['lon'], subset['lat'], c=CORES_REPS[c], s=70, alpha=0.8, label=f'Rep {c+1}', zorder=3)
for _, rep in df_reps.iterrows():
    axes[1].scatter(rep['lon_base'], rep['lat_base'], c='black', marker='*', s=200, zorder=5)
axes[1].set_title('Territórios por K-Means\n(cada cor = 1 representante, ★ = base)', fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].legend(loc='upper right', fontsize=8, ncol=2)

plt.suptitle('Clusterização Geográfica de Farmácias', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/01_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'🟣 K-Means + 2-Opt por Território:')
print(f'   Total: {total_cluster:.1f} km | Saving: {saving_cluster:.1f} km ({saving_cluster/total_aleatorio*100:.1f}%)')

## 8. Comparativo de Algoritmos

In [ ]:
# Tabela comparativa
algoritmos = {
    'Aleatório\n(atual)':           total_aleatorio,
    'Nearest\nNeighbor':            total_nn,
    '2-Opt':                        total_2opt,
    'Prioridade\n(NN)':             total_prior,
    'K-Means\n+ 2-Opt':            total_cluster,
}

df_comp = pd.DataFrame([
    {
        'Algoritmo': k,
        'Total km': v,
        'Saving km': total_aleatorio - v,
        'Saving %': (total_aleatorio - v) / total_aleatorio * 100,
        'Economia R$/sem': (total_aleatorio - v) * CUSTO_KM,
        'Economia R$/ano': (total_aleatorio - v) * CUSTO_KM * 50,
    } for k, v in algoritmos.items()
])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

nomes = list(algoritmos.keys())
totais = list(algoritmos.values())
cores_alg = [CORES['antes'], CORES['nn'], CORES['opt2'], CORES['prior'], CORES['cluster']]

# Distância total
bars = axes[0].bar(nomes, totais, color=cores_alg, edgecolor='white', width=0.6)
for bar, val in zip(bars, totais):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'{val:.0f}km', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Distância Total por Algoritmo (km)', fontweight='bold')
axes[0].set_ylabel('km')
axes[0].tick_params(axis='x', labelsize=8)

# Saving %
savings_pct = [0] + [(total_aleatorio - v) / total_aleatorio * 100 for v in list(algoritmos.values())[1:]]
bars2 = axes[1].bar(nomes, savings_pct, color=cores_alg, edgecolor='white', width=0.6)
for bar, val in zip(bars2, savings_pct):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Saving em % vs Rota Aleatória', fontweight='bold')
axes[1].set_ylabel('Saving (%)')
axes[1].tick_params(axis='x', labelsize=8)

# Economia anual
economias = [0] + [(total_aleatorio - v) * CUSTO_KM * 50 for v in list(algoritmos.values())[1:]]
bars3 = axes[2].bar(nomes, economias, color=cores_alg, edgecolor='white', width=0.6)
for bar, val in zip(bars3, economias):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'R${val:,.0f}', ha='center', fontsize=8, fontweight='bold')
axes[2].set_title('Economia Anual por Algoritmo (R$)', fontweight='bold')
axes[2].set_ylabel('R$')
axes[2].tick_params(axis='x', labelsize=8)

plt.suptitle('Comparativo de Algoritmos de Otimização de Rotas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/02_comparativo_algoritmos.png', dpi=150, bbox_inches='tight')
plt.show()

print(df_comp.round(1).to_string(index=False))

## 9. Planejamento Semanal Completo
Com capacidade máxima de 6 visitas por dia, distribuímos as farmácias em 5 dias de trabalho por representante.

In [ ]:
DIAS_SEMANA = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta']

def planejar_semana(rep_id, farm_indices, df_farm, base):
    """
    Distribui as farmácias em dias da semana:
    - Alta prioridade: priorizadas nos primeiros dias
    - Máximo MAX_VISITAS_DIA por dia
    - Rota diária otimizada com NN
    """
    farms_dia = {dia: [] for dia in DIAS_SEMANA}
    dist_dia  = {dia: 0 for dia in DIAS_SEMANA}

    # Ordena por prioridade
    farms_ord = df_farm.loc[farm_indices].sort_values('peso_prior', ascending=False)

    # Distribui
    idx_dia = 0
    for _, f in farms_ord.iterrows():
        dia = DIAS_SEMANA[idx_dia]
        farms_dia[dia].append(f['id'])
        if len(farms_dia[dia]) >= MAX_VISITAS_DIA:
            idx_dia = min(idx_dia + 1, len(DIAS_SEMANA) - 1)

    # Otimiza cada dia
    for dia in DIAS_SEMANA:
        if farms_dia[dia]:
            pontos = [(df_farm[df_farm['id']==fid]['lat'].values[0],
                       df_farm[df_farm['id']==fid]['lon'].values[0]) for fid in farms_dia[dia]]
            rota = nearest_neighbor(base, pontos)
            dist_dia[dia] = dist_rota(rota)

    return farms_dia, dist_dia

# Gera planejamento para todos os representantes
planejamento = {}
for i, rep in df_reps.iterrows():
    base = (rep['lat_base'], rep['lon_base'])
    farms_dia, dist_dia = planejar_semana(
        rep['id'], farm_idx_por_rep[i], df_farm, base
    )
    planejamento[rep['id']] = {'farms_dia': farms_dia, 'dist_dia': dist_dia}

# Visualização — heatmap de visitas por dia
heatmap_data = np.array([
    [len(planejamento[rep['id']]['farms_dia'][dia]) for dia in DIAS_SEMANA]
    for _, rep in df_reps.iterrows()
])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Greens',
            xticklabels=DIAS_SEMANA,
            yticklabels=[f"Rep {i+1}" for i in range(N_REPRESENTANTES)],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Farmácias por Representante/Dia\n(planejamento semanal)', fontweight='bold')

# Distância por dia
dist_por_dia = {dia: sum(planejamento[rep['id']]['dist_dia'][dia] for _, rep in df_reps.iterrows())
                for dia in DIAS_SEMANA}
axes[1].bar(DIAS_SEMANA, list(dist_por_dia.values()), color=CORES['nn'], edgecolor='white')
for dia, val in dist_por_dia.items():
    axes[1].text(list(DIAS_SEMANA).index(dia), val + 1, f'{val:.0f}km', ha='center', fontweight='bold')
axes[1].set_title('Distância Total da Equipe por Dia\n(soma de todos os representantes)', fontweight='bold')
axes[1].set_ylabel('km')

plt.suptitle('Planejamento Semanal — Capacidade Máxima: 6 visitas/dia', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/03_planejamento_semanal.png', dpi=150, bbox_inches='tight')
plt.show()

# Exemplo de agenda — Representante 1
rep1 = df_reps.iloc[0]['id']
print(f'\n📅 Agenda semanal — {rep1}:')
for dia in DIAS_SEMANA:
    farms = planejamento[rep1]['farms_dia'][dia]
    dist  = planejamento[rep1]['dist_dia'][dia]
    priors = [df_farm[df_farm['id']==f]['prioridade'].values[0] for f in farms]
    print(f'   {dia:8s}: {len(farms)} farmácias | {dist:.1f}km | {farms} | Prioridades: {priors}')

## 10. Heatmap de Cobertura
Visualiza a densidade de farmácias por região antes e depois da clusterização.

In [ ]:
# Heatmap de cobertura por região
cobertura_regiao = df_farm.groupby(['regiao', 'prioridade']).size().unstack(fill_value=0)
cobertura_fat    = df_farm.groupby('regiao')['faturamento'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Heatmap de contagem por região e prioridade
if not cobertura_regiao.empty:
    sns.heatmap(cobertura_regiao, annot=True, fmt='d', cmap='YlGn',
                ax=axes[0], linewidths=0.5)
    axes[0].set_title('Farmácias por Região e Prioridade', fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30, labelsize=9)

# Faturamento por região
bars = axes[1].barh(cobertura_fat.index, cobertura_fat.values / 1e6,
                    color=CORES['nn'], edgecolor='white')
for bar, val in zip(bars, cobertura_fat.values):
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'R${val/1e6:.1f}M', va='center', fontsize=9)
axes[1].set_title('Faturamento Mensal por Região (R$ Mi)', fontweight='bold')
axes[1].set_xlabel('R$ Milhões')

# Distribuição de prioridade por região
cobertura_pct = cobertura_regiao.div(cobertura_regiao.sum(axis=1), axis=0) * 100
cobertura_pct.plot(kind='bar', ax=axes[2], color=[CORES['antes'], CORES['opt2'], CORES['prior']],
                   edgecolor='white', width=0.7)
axes[2].set_title('% de Farmácias por Prioridade\npor Região', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=45, labelsize=8)
axes[2].legend(title='Prioridade', fontsize=8)

plt.suptitle('Análise de Cobertura Geográfica — Distribuidora Farmacêutica',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/04_heatmap_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Projeção de ROI — 12 Meses

In [ ]:
# Usa o melhor algoritmo (2-Opt) para a projeção
melhor_saving_sem = (total_aleatorio - total_2opt) * CUSTO_KM
custo_impl = 5000

meses = list(range(1, 13))
saving_mensal = [melhor_saving_sem * 4 * np.random.uniform(0.88, 1.12) for _ in meses]
saving_acum   = np.cumsum(saving_mensal)
roi_acum      = saving_acum - custo_impl
breakeven     = next((i+1 for i, r in enumerate(roi_acum) if r > 0), None)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Saving mensal
axes[0].bar(meses, saving_mensal, color=CORES['nn'], alpha=0.85, edgecolor='white')
axes[0].axhline(np.mean(saving_mensal), color=CORES['antes'], linestyle='--', linewidth=2,
                label=f'Média: R$ {np.mean(saving_mensal):,.0f}')
axes[0].set_title('Saving Mensal (R$)', fontweight='bold')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('R$')
axes[0].set_xticks(meses)
axes[0].set_xticklabels([f'M{m}' for m in meses])
axes[0].legend()

# ROI acumulado
axes[1].plot(meses, roi_acum, marker='o', color=CORES['nn'], linewidth=2.5, markersize=7)
axes[1].fill_between(meses, roi_acum, 0,
                     where=[r >= 0 for r in roi_acum], alpha=0.15, color=CORES['nn'])
axes[1].fill_between(meses, roi_acum, 0,
                     where=[r < 0 for r in roi_acum], alpha=0.15, color=CORES['antes'])
axes[1].axhline(0, color='gray', linewidth=1)
if breakeven:
    axes[1].axvline(breakeven, color=CORES['opt2'], linestyle=':', linewidth=2,
                    label=f'Break-even: Mês {breakeven}')
axes[1].set_title(f'ROI Acumulado 12 meses\nTotal: R$ {roi_acum[-1]:,.0f}', fontweight='bold')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('R$')
axes[1].set_xticks(meses)
axes[1].set_xticklabels([f'M{m}' for m in meses])
axes[1].legend()

# Comparativo de ROI por algoritmo
nomes_alg = ['NN', '2-Opt', 'Prioridade', 'K-Means\n+2-Opt']
savings_alg = [
    (total_aleatorio - total_nn) * CUSTO_KM * 50,
    (total_aleatorio - total_2opt) * CUSTO_KM * 50,
    (total_aleatorio - total_prior) * CUSTO_KM * 50,
    (total_aleatorio - total_cluster) * CUSTO_KM * 50,
]
bars = axes[2].bar(nomes_alg, savings_alg,
                   color=[CORES['nn'], CORES['opt2'], CORES['prior'], CORES['cluster']],
                   edgecolor='white')
for bar, val in zip(bars, savings_alg):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'R${val:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[2].set_title('Economia Anual por Algoritmo (R$)', fontweight='bold')
axes[2].set_ylabel('R$/ano')

plt.suptitle('Projeção de ROI — Otimização de Rotas Comerciais', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/05_roi_projecao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💰 Saving anual (2-Opt): R$ {saving_acum[-1]:,.0f}')
print(f'📈 ROI líquido 12 meses: R$ {roi_acum[-1]:,.0f}')
print(f'📅 Break-even: Mês {breakeven}')

## 12. Mapa Interativo — Antes vs Depois

In [ ]:
def criar_mapa_completo(rotas, distancias, titulo, arquivo):
    m = folium.Map(location=[-23.5505, -46.6333], zoom_start=11, tiles='CartoDB positron')

    titulo_html = f"""<div style="position:fixed;top:10px;left:50%;
        transform:translateX(-50%);z-index:1000;background:white;
        padding:10px 20px;border-radius:8px;border:2px solid #333;
        font-size:14px;font-weight:bold;text-align:center;">{titulo}</div>"""
    m.get_root().html.add_child(folium.Element(titulo_html))

    farm_rep = {}
    for i, rep in df_reps.iterrows():
        for idx in farm_idx_por_rep[i]:
            farm_rep[idx] = (rep['id'], rep['nome'], CORES_REPS[i])

    for idx, f in df_farm.iterrows():
        rep_id, rep_nome, cor_rep = farm_rep.get(idx, ('?', '?', 'gray'))
        icone = {'Alta': '🔴', 'Media': '🟠', 'Baixa': '🔵'}[f['prioridade']]
        cor   = {'Alta': 'red', 'Media': 'orange', 'Baixa': 'blue'}[f['prioridade']]
        dia_visita = ''
        for dia in DIAS_SEMANA:
            if f['id'] in planejamento.get(rep_id, {}).get('farms_dia', {}).get(dia, []):
                dia_visita = dia
                break
        popup_html = f"""
        <div style="font-family:Arial;font-size:13px;min-width:210px;">
            <b>{icone} {f['nome']}</b><hr style="margin:4px 0">
            <b>Prioridade:</b> {f['prioridade']}<br>
            <b>Pedidos/mês:</b> {f['pedidos_mes']}<br>
            <b>Faturamento:</b> R$ {f['faturamento']:,.0f}<br>
            <b>Representante:</b> <span style="color:{cor_rep}">■</span> {rep_nome}<br>
            <b>Visita:</b> {dia_visita}
        </div>"""
        folium.CircleMarker(
            location=[f['lat'], f['lon']],
            radius=7 if f['prioridade'] == 'Alta' else 5,
            color=cor, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{f['id']} | {f['prioridade']} | {rep_id} | {dia_visita}"
        ).add_to(m)

    for i, (rep_id, rota) in enumerate(rotas.items()):
        cor  = CORES_REPS[i % len(CORES_REPS)]
        dist = distancias[rep_id]
        rep_nome = df_reps[df_reps['id'] == rep_id]['nome'].values[0]
        folium.PolyLine(rota, color=cor, weight=2.5, opacity=0.75,
                        tooltip=f"{rep_nome} | {dist:.1f} km").add_to(m)
        folium.Marker(rota[0],
                      icon=folium.Icon(color='black', icon='home', prefix='fa'),
                      popup=folium.Popup(f"<b>🏠 {rep_nome}</b><br>{dist:.1f} km", max_width=180),
                      tooltip=f"Base: {rep_nome}").add_to(m)

    legenda_reps = ''.join([
        f"<span style='color:{CORES_REPS[i]};font-size:15px;'>■</span> "
        f"{row['nome']}: <b>{distancias[row['id']]:.1f}km</b><br>"
        for i, (_, row) in enumerate(df_reps.iterrows())
    ])
    dist_total = sum(distancias.values())

    m.get_root().html.add_child(folium.Element(f"""
    <div style="position:fixed;bottom:20px;left:15px;z-index:1000;
                background:white;padding:14px;border-radius:10px;
                border:2px solid #ccc;font-size:12px;max-width:220px;">
        <b>📋 Representantes</b><br><br>{legenda_reps}
        <hr style="margin:6px 0"><b>Total: {dist_total:.1f} km</b>
    </div>
    <div style="position:fixed;bottom:20px;right:15px;z-index:1000;
                background:white;padding:14px;border-radius:10px;
                border:2px solid #ccc;font-size:13px;">
        <b>🏥 Prioridade</b><br><br>
        🔴 Alta (visita 1ª)<br>🟠 Média<br>🔵 Baixa<br>🏠 Base
    </div>"""))

    m.save(f'mapas/{arquivo}')
    print(f'✅ Mapa salvo: mapas/{arquivo}')
    return m

mapa_antes  = criar_mapa_completo(rotas_aleatorio, dist_aleatorio, 'ANTES — Rota Manual (Aleatória)', 'antes.html')
mapa_depois = criar_mapa_completo(rotas_2opt, dist_2opt, 'DEPOIS — Rota Otimizada (2-Opt)', 'depois.html')
print('\n📍 Abra os HTMLs no navegador para explorar os mapas interativos!')
mapa_depois

## 13. Dashboard Final — Visão Executiva

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Gráfico 1: Comparativo de algoritmos
ax1 = fig.add_subplot(gs[0, :])
nomes_full = ['Aleatório\n(atual)', 'Nearest\nNeighbor', '2-Opt', 'Prioridade', 'K-Means\n+2-Opt']
totais_full = [total_aleatorio, total_nn, total_2opt, total_prior, total_cluster]
cores_full  = [CORES['antes'], CORES['nn'], CORES['opt2'], CORES['prior'], CORES['cluster']]
bars = ax1.bar(nomes_full, totais_full, color=cores_full, edgecolor='white', width=0.55)
ax1.axhline(total_aleatorio, color=CORES['antes'], linestyle='--', alpha=0.5, linewidth=1.5)
for bar, val, total in zip(bars, totais_full, totais_full):
    saving = total_aleatorio - val
    label  = f'{val:.0f} km' if val == total_aleatorio else f'{val:.0f} km\n(−{saving:.0f}km)'
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, label,
             ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Comparativo de Distância Total por Algoritmo (km/semana)', fontweight='bold', fontsize=12)
ax1.set_ylabel('km')

# Gráfico 2: Saving por representante (2-Opt)
ax2 = fig.add_subplot(gs[1, 0:2])
reps_ids = [rep['id'] for _, rep in df_reps.iterrows()]
savings_rep = [dist_aleatorio[r] - dist_2opt[r] for r in reps_ids]
cores_saving = [CORES['nn'] if s > 0 else CORES['antes'] for s in savings_rep]
bars2 = ax2.bar(reps_ids, savings_rep, color=cores_saving, edgecolor='white')
ax2.axhline(np.mean(savings_rep), color='black', linestyle='--', linewidth=1.5,
            label=f'Média: {np.mean(savings_rep):.1f} km')
ax2.set_title('Saving por Representante — 2-Opt (km)', fontweight='bold')
ax2.set_ylabel('km economizados')
ax2.tick_params(axis='x', rotation=45, labelsize=8)
ax2.legend()

# Gráfico 3: Pizza de prioridade
ax3 = fig.add_subplot(gs[1, 2])
pri_counts = df_farm['prioridade'].value_counts()
ax3.pie(pri_counts, labels=pri_counts.index, autopct='%1.1f%%',
        colors=[CORES['antes'], CORES['opt2'], CORES['prior']],
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}, startangle=90)
ax3.set_title('Farmácias por\nPrioridade', fontweight='bold')

# Gráfico 4: ROI acumulado
ax4 = fig.add_subplot(gs[2, 0:2])
ax4.plot(meses, roi_acum, marker='o', color=CORES['nn'], linewidth=2.5, markersize=7)
ax4.fill_between(meses, roi_acum, 0, where=[r >= 0 for r in roi_acum], alpha=0.15, color=CORES['nn'])
ax4.fill_between(meses, roi_acum, 0, where=[r < 0 for r in roi_acum],  alpha=0.15, color=CORES['antes'])
ax4.axhline(0, color='gray', linewidth=1)
if breakeven:
    ax4.axvline(breakeven, color=CORES['opt2'], linestyle=':', linewidth=2, label=f'Break-even: Mês {breakeven}')
ax4.set_title(f'ROI Acumulado 12 Meses (R$)', fontweight='bold')
ax4.set_xlabel('Mês')
ax4.set_ylabel('R$')
ax4.set_xticks(meses)
ax4.set_xticklabels([f'M{m}' for m in meses])
ax4.legend()

# Painel de KPIs
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')
kpis = [
    ('💊 Farmácias',          f'{N_FARMACIAS}'),
    ('👥 Representantes',     f'{N_REPRESENTANTES}'),
    ('📏 Distância antes',    f'{total_aleatorio:.0f} km'),
    ('📏 Melhor algoritmo',   f'{total_2opt:.0f} km (2-Opt)'),
    ('✅ Saving km',          f'{total_aleatorio-total_2opt:.0f} km ({(total_aleatorio-total_2opt)/total_aleatorio*100:.1f}%)'),
    ('💰 Saving/semana',      f'R$ {(total_aleatorio-total_2opt)*CUSTO_KM:,.0f}'),
    ('💰 Saving/ano',         f'R$ {(total_aleatorio-total_2opt)*CUSTO_KM*50:,.0f}'),
    ('📅 Break-even',         f'Mês {breakeven}'),
    ('📈 ROI 12 meses',       f'R$ {roi_acum[-1]:,.0f}'),
    ('🗓️ Visitas/dia/rep',   f'Até {MAX_VISITAS_DIA} farmácias'),
]
ax5.text(0.05, 0.97, '📊 KPIs do Projeto', transform=ax5.transAxes,
         fontsize=12, fontweight='bold', va='top')
for j, (label, valor) in enumerate(kpis):
    cor = CORES['nn'] if any(k in label for k in ['Saving', 'ROI', 'Break', 'Melhor']) else '#333'
    ax5.text(0.02, 0.88 - j*0.085, f'{label}:', transform=ax5.transAxes,
             fontsize=9, va='top', color='gray')
    ax5.text(0.55, 0.88 - j*0.085, valor, transform=ax5.transAxes,
             fontsize=9, va='top', fontweight='bold', color=cor)

fig.suptitle('Dashboard Executivo — Otimização de Rotas Comerciais | Distribuidora Farmacêutica | Grande São Paulo',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('images/06_dashboard_executivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Dashboard salvo em images/06_dashboard_executivo.png')

---
## 📋 Conclusões

| Algoritmo | Distância Total | Saving vs Baseline | Economia Anual |
|-----------|----------------|--------------------|----------------|
| Aleatório (atual) | 664 km | — | — |
| Nearest Neighbor | 539 km | 18,9% | R$ 9.386 |
| **2-Opt** | **518 km** | **22,0%** | **R$ 10.979** |
| Prioridade (NN) | 607 km | 8,6% | R$ 4.301 |
| **K-Means + 2-Opt** | **281 km** | **57,6%** | **R$ 28.689** |

> 📌 **Melhor resultado geral:** K-Means + 2-Opt com territórios geográficos coesos — redução de 57,6% na distância percorrida.  
> 📌 **Melhor heurística pura:** 2-Opt com 22% de saving e ROI positivo a partir do **Mês 6** (investimento inicial de R$ 5.000).

### Lições aprendidas

1. **Rotas no feeling custam caro** — 664 km/semana aleatório vs 281 km com K-Means: **383 km desperdiçados toda semana**
2. **Territorialização é o maior ganho** — K-Means elimina cruzamento de rotas entre representantes, gerando 57,6% de saving, muito além do 2-Opt isolado (22%)
3. **2-Opt supera NN** — a melhoria iterativa encontra soluções 3,1% mais curtas que a heurística gulosa, com custo computacional baixo
4. **Prioridade tem custo geográfico** — o algoritmo de prioridade (8,6% saving) sacrifica eficiência de km para garantir que farmácias de alto valor sejam visitadas primeiro; válido quando o risco de perder clientes estratégicos supera o custo de deslocamento
5. **Planejamento semanal operacional** — com 1 farmácia/dia por representante, a agenda fica previsível e o representante sabe exatamente o que fazer ao acordar
6. **ROI rápido** — break-even no Mês 6 com ROI líquido de R$ 5.373 em 12 meses (usando 2-Opt); com K-Means o retorno seria ainda maior

### 🚀 Próximos Passos

- **OR-Tools (Google):** solver exato para resultados ainda melhores que o 2-Opt
- **Google Maps API:** distâncias reais considerando trânsito e janelas de horário
- **Streamlit:** dashboard web para atualização dinâmica das rotas pelo gestor
- **Dados reais:** integração com CRM e sistema de pedidos para scoring de prioridade dinâmico